2026.11.20. Same sections as Pred_visual_npc_uni2_A2.ipynb, for this arm's five-fold mean on patient1. Restart Kernel & Run All. Prediction is not repeated.


# NPC external validation — C  tile_local_global_k8

This notebook shows **patient1** only (`20210809-C-AH4199551`). It follows `Pred_visual_npc_uni2_A2.ipynb`: composition, spatial maps, Visium resolution, per-type abundance, TLS, and niche indices.

The weights are the 2026-11-20 arm **C  tile_local_global_k8**: the same two tokens as B, plus the mean of eight neighbours. Pooled OOF L2 macro AUROC on the training cohorts was 0.9376. Folds are stratified by section, seed 42, cohort-safe inner validation.

The maps are the **five-fold mean** already written under `data/Visium/NPC/pred_uni2_ens5_tile_local_global_k8_stratsec_ivsafe_seed42/`. They are not `best_mlp_gpu.pt` alone. This notebook does not launch prediction. `STAR_PROB_MIN` is 0, so every StarDist nucleus is kept.

Training OOF: `data/PanCancer/result_uni2_ablation_tile_local_global_k8_stratsec_new1066axisfix_ivsafe_paired_v2_seed42/oof_predictions.npz`.


## Predictions already written

These cells only draw `pred_uni2_ens5_tile_local_global_k8_stratsec_ivsafe_seed42`. Do not point `demo_uni2.sh` at this checkpoint: that script builds 14 px UNI2 tokens, and this arm was trained on tile features.


In [ ]:
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams["pdf.fonttype"] = 42  # TrueType; PDF text stays editable
mpl.rcParams["ps.fonttype"] = 42

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
# 2026.10.28, explicit checkout override / Farm 可设 HIST2PHENO_REPO
import os
if os.environ.get("HIST2PHENO_REPO"):
    REPO = Path(os.environ["HIST2PHENO_REPO"]).expanduser().resolve()
if not (REPO / "code").is_dir():
    REPO = Path("/nobackup2/users/lingyu/Python/Hist2Pheno")
sys.path.insert(0, str(REPO / "code"))
sys.path.insert(0, str(REPO / "code" / "External_val" / "NPC"))

from Hist2Pheno_pkg.external_validation import ExternalCohort
from npc_figures import (
    C2L_TYPES,
    STACK_ORDER,
    c2l_palette,
    configure_figure_dpi,
    draw_spot_circles,
    draw_tls,
    group_fraction,
    keep_leftmost_foci,
    legend_handles,
    load_tls_xy,
    map_l2,
    nuclei_in_contours,
    save_figure,
    tls_score_row,
    tls_smooth_contours,
)

import numpy as np

OOF_NPZ = (
    REPO
    / "data/PanCancer/result_uni2_ablation_tile_local_global_k8_stratsec_new1066axisfix_ivsafe_paired_v2_seed42"
    / "oof_predictions.npz"
)
CKPT = (
    REPO
    / "data/PanCancer/result_uni2_ablation_tile_local_global_k8_stratsec_new1066axisfix_ivsafe_paired_v2_seed42"
    / "cross_dataset_cv/tile_local_global_k8/best_mlp_gpu.pt"
)

STAR_PROB_MIN = 0.00
import os as _prob_os
_prob_os.environ["STAR_PROB_MIN"] = "0"
# Five-fold mean already written by predict_finished_arms.py. Do not re-predict:
# a tile checkpoint cannot be run through the 14 px UNI2 feature path.
pred_dirname = "pred_uni2_ens5_tile_local_global_k8_stratsec_ivsafe_seed42"
_cohort = ExternalCohort(
    name="NPC",
    data_root=REPO / "data" / "Visium" / "NPC",
    repo=REPO,
    patch_size=14,
    method="UNI2",
    embedding_dirname="project_all_UNI2",
    pred_dirname=pred_dirname,
    checkpoint=CKPT,
    project_tag="project_all_UNI2",
)
PRED_ROOT = _cohort.pred_root
composition_table = _cohort.composition_table
discover_samples = _cohort.discover_samples
geometry = _cohort.geometry
sample_paths = _cohort.sample_paths
ensure_l2_probs = _cohort.ensure_l2_probs
plot_spatial = _cohort.plot_spatial
plot_composition = _cohort.plot_composition
SAVE_FIGURES = True
FIGSIZE = (17.2, 6.6)
DPI = 200
SAVE_DPI = 300
configure_figure_dpi(save_dpi=SAVE_DPI, plot_dpi=DPI)
HE_MAX_SIDE = 1400
MARKER_PX = 1.2
samples = [s for s in discover_samples() if s == "20210809-C-AH4199551"]
print("notebook shows patient1 only:", samples)
if not samples:
    raise RuntimeError("patient1 20210809-C-AH4199551 was not discovered")
print("C  tile_local_global_k8: five-fold mean, StarDist probability >= 0")
print(f"OOF exists={OOF_NPZ.is_file()}  {OOF_NPZ}")
print(f"checkpoint {CKPT.name} exists={CKPT.is_file()}  (plots use the five-fold mean, not this single fold)")
print(f"pred root {PRED_ROOT}")
for sample in samples:
    g = geometry(sample)
    print(
        f"{sample}  mpp={g['mpp_um']:.4f} µm/px  "
        f"native14={g['patch_um_if_native_16px']:.2f} µm  "
        f"after rescale={g['patch_um_after_rescale']:.1f} µm  "
        f"scale={g['scale']:.4f}"
    )
    p = sample_paths(sample)
    print(f"  embeddings: {p['emb']}")
    print(f"  pred exists: {p['pred'].is_file()}  {p['pred']}")
    if not p["pred"].is_file() or not p["pred"].with_name(f"{sample}_l2_probs.npz").is_file():
        raise FileNotFoundError(
            "Five-fold prediction is missing. Run: "
            "CUDA_VISIBLE_DEVICES=0 python code/External_val/NPC/predict_finished_arms.py"
        )

# The ensemble h5ad has no single-checkpoint .pipeline.json. Read it directly.
import importlib
_prov = importlib.import_module("Hist2Pheno_pkg.pipeline_provenance")
_cons = importlib.import_module("Hist2Pheno_pkg.external_consistency")
_prov.read_verified_prediction = lambda path: __import__("anndata").read_h5ad(path)
def _ensemble_probs(cohort, sample, *, force=False, allow_cpu=False):
    del force, allow_cpu
    path = cohort.l2_probs_path(sample)
    with np.load(path, allow_pickle=True) as z:
        return np.asarray(z["probs"], dtype="float32").copy(), [str(x) for x in z["class_names"]]
_cons.preflight = lambda cohort, samples, *, allow_cpu=False: None
_cons.probabilities = _ensemble_probs


In [ ]:
import pandas as pd
from IPython.display import Image, display

missing = [s for s in samples if not sample_paths(s)["pred"].is_file()]
if missing:
    raise FileNotFoundError(
        "No UNI2 predictions yet for "
        + ", ".join(missing)
        + ". The five-fold h5ad should already be under the pred root."
    )

comp = composition_table(samples)

for head in ("L1", "L12", "L2"):
    sub = comp.loc[comp["head"] == head].copy()
    order = sub.groupby("class")["fraction"].max().sort_values(ascending=False).index
    wide = (
        sub.pivot_table(index="class", columns="sample", values=["fraction", "n", "mean_conf"])
        .reindex(order)
    )
    print(f"\n===== {head} =====")
    display(wide.round(3))


In [ ]:
import matplotlib.pyplot as plt

for fig in plot_spatial(
    samples,
    figsize=FIGSIZE,
    dpi=DPI,
    he_max_side=HE_MAX_SIDE,
    marker_px=MARKER_PX,
    save=SAVE_FIGURES,
    save_dpi=SAVE_DPI,
):
    display(fig)
    plt.close(fig)


## Resolution from Visium spot to single cell

Same mapping as `Fig2_Fig3_panels.ipynb` (Fig. 3b). Each StarDist nucleus goes to the **nearest** Visium spot and is kept only inside that spot’s capture disc (`spot_diameter_fullres / 2`), via `Hist2Pheno_pkg/nuclei_to_spots.py`.

Three spatial panels: **single-nucleus L2**, **pseudo-spot** (majority predicted L2 in the disc), **cell2location** (argmax of the 7 abundances). Patient1 uses `Annotation/sp.obs.csv` (`AH4199551`); patient2 uses `Annotation/AJ_sp.obs.csv` (`AJ3037946`). The next cell stacks Hist2Pheno vs pseudo-spot vs cell2location on the same 7-type vocabulary.


In [ ]:
import json

import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.spatial import cKDTree

from Hist2Pheno_pkg.nuclei_to_spots import nuclei_to_spots
from Hist2Pheno_pkg.plotting_palettes import get_palette

L2_PALETTE = get_palette("pancancer", "fine")
L2_NAMES = list(L2_PALETTE)
SAMPLE_TO_VISIUM = {
    "20210809-C-AH4199551": "patient1",
    "20210809-D-AJ3037946": "patient2",
}
SAMPLE_TO_C2L = {
    "20210809-C-AH4199551": "AH4199551",
    "20210809-D-AJ3037946": "AJ3037946",
}
NPC_DATA = REPO / "data" / "Visium" / "NPC"
C2L_FILES = [
    NPC_DATA / "Annotation" / "sp.obs.csv",
    NPC_DATA / "Annotation" / "AJ_sp.obs.csv",
]
fig_dir = PRED_ROOT / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

# cell2location types, coloured like the matching pan-cancer L2 classes.
C2L_COLOR = c2l_palette(L2_PALETTE)


missing = [p for p in C2L_FILES if not p.is_file()]
if missing:
    raise FileNotFoundError(missing)
c2l_obs = pd.concat([pd.read_csv(p, index_col=0) for p in C2L_FILES], axis=0)
c2l_obs.index = c2l_obs.index.astype(str)
print(
    "cell2location",
    [p.name for p in C2L_FILES],
    "rows",
    len(c2l_obs),
    "samples",
    sorted(c2l_obs["sample"].astype(str).unique()),
)

nucleus_s = max(0.12, (float(MARKER_PX) * 72.0 / float(DPI)) ** 2 * 0.35)
spot_maps = {}

for sample in samples:
    visium_id = SAMPLE_TO_VISIUM.get(sample)
    if visium_id is None:
        print(f"[skip] {sample}: no patient1/patient2 Visium folder mapping")
        continue
    npc_dir = NPC_DATA / visium_id
    pred_path = sample_paths(sample)["pred"]
    if not pred_path.is_file():
        raise FileNotFoundError(pred_path)
    scale = json.loads((npc_dir / "scalefactors_json.json").read_text())
    radius_px = float(scale["spot_diameter_fullres"]) / 2
    pos = pd.read_csv(
        npc_dir / "tissue_positions_list.csv",
        header=None,
        names=["barcode", "in_tissue", "array_row", "array_col", "pxl_row", "pxl_col"],
    )
    pos = pos.loc[pos["in_tissue"] == 1].reset_index(drop=True)
    pos["barcode"] = pos["barcode"].astype(str)
    spot_xy = pos[["pxl_col", "pxl_row"]].to_numpy(dtype=np.float64)
    pitch_px = float(np.median(cKDTree(spot_xy).query(spot_xy, k=2)[0][:, 1]))
    npc = ad.read_h5ad(pred_path)
    npc_xy = np.asarray(npc.obsm["spatial_HE"], dtype=np.float64)
    npc_lab = npc.obs["pred_CT"].astype(str).to_numpy()
    captured = nuclei_to_spots(
        npc_xy, spot_xy, radius_px, shape="circle",
        labels=npc_lab, class_names=L2_NAMES,
    )
    counts = captured["n_nuclei"]
    used = counts > 0
    npc_spots = pos.loc[used].copy()
    npc_spots["pixel_x"] = spot_xy[used, 0]
    npc_spots["pixel_y"] = spot_xy[used, 1]
    npc_spots["n_nuclei"] = counts[used]
    npc_spots["dominant_l2"] = captured["dominant"][used]
    npc_spots["dominant_group"] = map_l2(npc_spots["dominant_l2"])
    for i, name in enumerate(L2_NAMES):
        npc_spots[f"n_{name}"] = captured["class_counts"][used, i]
        npc_spots[f"fraction_{name}"] = captured["class_counts"][used, i] / counts[used]
    npc_spots.to_csv(fig_dir / f"{sample}_visium_composition.csv", index=False)

    nuc_group = map_l2(npc_lab)
    h2p_frac = group_fraction(nuc_group)
    pseudo_frac = group_fraction(npc_spots["dominant_group"].to_numpy())

    c2l_id = SAMPLE_TO_C2L.get(sample)
    c2l_xy = c2l_lab = None
    c2l_frac = None
    if c2l_id is not None:
        sub = c2l_obs.loc[c2l_obs["sample"].astype(str) == str(c2l_id)].copy()
        hit = pos["barcode"].isin(sub.index)
        aligned = sub.reindex(pos.loc[hit, "barcode"])
        raw = aligned[list(C2L_TYPES)].to_numpy(dtype=np.float64)
        ok = np.isfinite(raw).all(axis=1) & (raw.sum(axis=1) > 0)
        raw = raw[ok]
        c2l_xy = spot_xy[hit.to_numpy()][ok]
        c2l_lab = np.array(C2L_TYPES)[np.argmax(raw, axis=1)]
        c2l_frac = group_fraction(c2l_lab)
        print(
            f"cell2location {c2l_id}: {int(ok.sum()):,} / {int(hit.sum()):,} "
            f"in-tissue spots matched"
        )

    spot_maps[sample] = {
        "h2p": h2p_frac,
        "pseudo": pseudo_frac,
        "c2l": c2l_frac,
        "n_nuc": len(npc_lab),
        "n_spot": int(used.sum()),
        "n_c2l": 0 if c2l_lab is None else int(len(c2l_lab)),
    }
    frac = pd.DataFrame(
        {
            "group": list(STACK_ORDER),
            "Hist2Pheno": [h2p_frac[k] for k in STACK_ORDER],
            "pseudo_spot": [pseudo_frac[k] for k in STACK_ORDER],
            "cell2location": [
                float(c2l_frac[k]) if c2l_frac is not None else np.nan
                for k in STACK_ORDER
            ],
        }
    )
    frac.to_csv(fig_dir / f"{sample}_nuclei_vs_visium_fraction.csv", index=False)
    display(frac.round(4))
    print(
        f"NPC {sample} ({visium_id}): nuclei {len(npc_xy):,}, "
        f"in-tissue spots {len(spot_xy):,}, pitch {pitch_px:.1f} px, "
        f"capture radius {radius_px:.1f} px, "
        f"nuclei in a spot {int(captured['inside'].sum()):,} / {len(npc_xy):,}, "
        f"spots with nuclei {int(used.sum()):,}"
    )

    spot_draw_radius = 0.50 * pitch_px
    fig, axes = plt.subplots(1, 3, figsize=(16.8, 5.6), dpi=DPI)
    axes[0].set_title("Hist2Pheno  single nuclei")
    for name in L2_NAMES:
        mask = npc_lab == name
        if np.any(mask):
            axes[0].scatter(
                npc_xy[mask, 0], npc_xy[mask, 1], s=nucleus_s,
                c=[L2_PALETTE[name]], linewidths=0, rasterized=True,
            )
    draw_spot_circles(
        axes[1], spot_xy[used], npc_spots["dominant_l2"].to_numpy(),
        spot_draw_radius, L2_PALETTE, L2_NAMES,
    )
    axes[1].set_title("Pseudo-spot  (majority L2)")
    if c2l_xy is None:
        axes[2].text(0.5, 0.5, "no cell2location\nfor this slide", ha="center", va="center")
        axes[2].set_title("cell2location")
    else:
        draw_spot_circles(
            axes[2], c2l_xy, c2l_lab, spot_draw_radius, C2L_COLOR, C2L_TYPES,
        )
        axes[2].set_title("cell2location  (argmax abundance)")
    pad = pitch_px
    for ax in axes:
        ax.set_xlim(npc_xy[:, 0].min() - pad, npc_xy[:, 0].max() + pad)
        ax.set_ylim(npc_xy[:, 1].max() + pad, npc_xy[:, 1].min() - pad)
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
    present_l2 = [
        n for n in L2_NAMES
        if n in set(npc_lab) or n in set(npc_spots["dominant_l2"].astype(str))
    ]
    axes[1].legend(
        handles=legend_handles(present_l2, L2_PALETTE),
        frameon=False, fontsize=9, loc="upper center",
        bbox_to_anchor=(0.5, -0.04), ncol=4,
    )
    if c2l_lab is not None:
        present_c2l = [n for n in C2L_TYPES if n in set(c2l_lab)]
        axes[2].legend(
            handles=legend_handles(present_c2l, C2L_COLOR, markersize=9),
            frameon=False, fontsize=10, loc="upper center",
            bbox_to_anchor=(0.5, -0.04), ncol=4,
        )
    fig.suptitle(f"NPC {sample}  nuclei / pseudo-spot / cell2location", fontsize=12, y=1.02)
    fig.tight_layout()
    fig.subplots_adjust(bottom=0.16)
    if SAVE_FIGURES:
        save_figure(fig, fig_dir / f"{sample}_nuclei_vs_visium.pdf")
    display(fig)
    plt.close(fig)


In [ ]:
# 2026-11-15. Per-type abundance, not argmax.
# Hist2Pheno: mean L2 softmax of nuclei inside each Visium spot, summed into the
# 7 cell2location types. cell2location: expected cell counts (sp.obs).
# Each panel uses its own 99th-percentile scale, so a large type does not hide a small one.
from npc_figures import l2_group_means, load_aligned_l2_probs, plot_abundance_maps
from scipy.spatial import cKDTree

for sample in samples:
    visium_id = SAMPLE_TO_VISIUM.get(sample)
    c2l_id = SAMPLE_TO_C2L.get(sample)
    if visium_id is None or c2l_id is None:
        print(f"[skip] {sample}: no Visium or cell2location id")
        continue
    pred_path = sample_paths(sample)["pred"]
    npc_xy, probs, class_names = load_aligned_l2_probs(
        pred_path, pred_path.with_name(f"{sample}_l2_probs.npz"),
    )
    npc_dir = NPC_DATA / visium_id
    scale = json.loads((npc_dir / "scalefactors_json.json").read_text())
    radius_px = float(scale["spot_diameter_fullres"]) / 2
    pos = pd.read_csv(
        npc_dir / "tissue_positions_list.csv",
        header=None,
        names=["barcode", "in_tissue", "array_row", "array_col", "pxl_row", "pxl_col"],
    )
    pos = pos.loc[pos["in_tissue"] == 1].reset_index(drop=True)
    pos["barcode"] = pos["barcode"].astype(str)
    spot_xy = pos[["pxl_col", "pxl_row"]].to_numpy(dtype=np.float64)
    captured = nuclei_to_spots(
        npc_xy, spot_xy, radius_px, shape="circle", values=probs,
    )
    h2p = l2_group_means(captured["mean"], class_names)
    h2p[captured["n_nuclei"] == 0] = np.nan
    sub = c2l_obs.loc[c2l_obs["sample"].astype(str) == str(c2l_id)].copy()
    aligned = sub.reindex(pos["barcode"])
    c2l = aligned[list(C2L_TYPES)].to_numpy(dtype=np.float64)
    c2l[~np.isfinite(c2l).all(axis=1) | ~(np.nansum(c2l, axis=1) > 0)] = np.nan
    pitch_px = float(np.median(cKDTree(spot_xy).query(spot_xy, k=2)[0][:, 1]))
    fig, tab = plot_abundance_maps(
        spot_xy, h2p, c2l, list(C2L_TYPES),
        radius=0.50 * pitch_px, sample=sample, dpi=DPI,
    )
    tab.to_csv(fig_dir / f"{sample}_abundance_spearman.csv", index=False)
    display(tab.round(3))
    print(
        f"NPC {sample}: spots {len(spot_xy):,}, "
        f"with nuclei {int((captured['n_nuclei'] > 0).sum()):,}, "
        f"with cell2location {int(np.isfinite(c2l).all(axis=1).sum()):,}"
    )
    if SAVE_FIGURES:
        save_figure(fig, fig_dir / f"{sample}_h2p_vs_c2l_abundance.pdf")
    display(fig)
    plt.close(fig)


### Stacked composition (same 7 types)

All three bars are **class proportions** on the same 7 cell2location types. Hist2Pheno = fraction of nuclei after mapping L2 → these types. Pseudo-spot = fraction of spots by majority mapped L2. cell2location = fraction of spots after converting per-spot abundances to a class (`argmax`). `other` is Endothelial / Neural (Hist2Pheno only).


In [ ]:
# 100% stacked class proportions: Hist2Pheno nuclei vs pseudo-spot majority vs cell2location argmax.
# Groups are the 7 cell2location types (+ other). Patient1: sp.obs.csv; patient2: AJ_sp.obs.csv.
methods = ("Hist2Pheno", "pseudo-spot", "cell2location")
method_key = {"Hist2Pheno": "h2p", "pseudo-spot": "pseudo", "cell2location": "c2l"}
samples_plot = [s for s in samples if s in spot_maps]
n_panel = max(len(samples_plot), 1)
fig, axes = plt.subplots(
    1, n_panel, figsize=(1.2 * n_panel + 1.8, 3.0), dpi=DPI, squeeze=False,
)
axes = axes[0]
for ax, sample in zip(axes, samples_plot):
    rec = spot_maps[sample]
    x = np.arange(len(methods))
    bottom = np.zeros(len(methods))
    for grp in STACK_ORDER:
        vals = []
        for m in methods:
            ser = rec[method_key[m]]
            vals.append(float(ser[grp]) if ser is not None else 0.0)
        vals = np.asarray(vals, dtype=float)
        if np.allclose(vals, 0.0):
            continue
        ax.bar(
            x, vals, bottom=bottom, width=0.62,
            color=C2L_COLOR[grp][:3], edgecolor="white", linewidth=0.4, label=grp,
        )
        bottom = bottom + vals
    ax.set_xticks(x)
    ax.set_xticklabels(methods, rotation=20, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("fraction")
    extra = ""
    if rec["c2l"] is None:
        extra = "  (no cell2location)"
        ax.text(2, 0.5, "n/a", ha="center", va="center", fontsize=9, color="#888888")
    ax.set_title(f"{sample}{extra}", fontsize=10)
    ax.set_xlim(-0.55, 2.55)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels, loc="center left", bbox_to_anchor=(1.01, 0.5),
    frameon=False, fontsize=10, title="mapped type",
)
fig.suptitle("NPC  Hist2Pheno vs pseudo-spot vs cell2location", fontsize=12, y=1.02)
fig.tight_layout()
if SAVE_FIGURES:
    save_figure(fig, fig_dir / "NPC_hist2pheno_pseudospot_cell2location_stacked.pdf")
display(fig)
plt.close(fig)

rows = []
for sample, rec in spot_maps.items():
    for method, key in method_key.items():
        ser = rec[key]
        if ser is None:
            continue
        for grp in STACK_ORDER:
            rows.append(
                {"sample": sample, "method": method, "group": grp, "fraction": float(ser[grp])}
            )
stack_tab = pd.DataFrame(rows)
stack_path = fig_dir / "NPC_hist2pheno_pseudospot_cell2location_stacked.csv"
stack_tab.to_csv(stack_path, index=False)
print(stack_path)
display(stack_tab.pivot_table(index="group", columns=["sample", "method"], values="fraction").round(3))


## TLS annotation (patient1)

Expert Visium TLS barcodes in `data/Visium/NPC/Annotation/TLS_spot_label.csv` (36 spots on `AH4199551` / `20210809-C-AH4199551`). Each barcode is mapped through `patient1/tissue_positions_list.csv`. A Gaussian-smoothed capture disc around those spots gives a **smooth outline** of each TLS focus. **Only the three left-side foci are kept** (the two lower-right clusters are dropped). The zoom box is the bounding rectangle of the kept spots plus padding. Overview: black outline only. Zoom: black outline + Visium spots. Then Hist2Pheno L1 / L12 / L2 inside that zoom. The L1 / L12 / L2 legend lists only nuclei inside the black TLS outlines, not every nucleus in the zoom rectangle. A 100% stacked bar uses that same outline set.


In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from matplotlib.patches import Rectangle

from Hist2Pheno_pkg.panel_figures import (
    crop_he_rectangle,
    nuclei_in_rectangle,
    plot_pred_composition_stacked,
    plot_roi_pred_spatial,
)

TLS_SAMPLE = "20210809-C-AH4199551"
NPC_DATA = REPO / "data" / "Visium" / "NPC"
TLS_PATH = NPC_DATA / "Annotation" / "TLS_spot_label.csv"
VISIUM_DIR = NPC_DATA / "patient1"
fig_dir = PRED_ROOT / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

ROI_BOX = "#E6B800"
PAD_PITCH = 2.6
N_KEEP_LEFT = 3

tls_xy, tls_bc, pitch_px, spot_r_px = load_tls_xy(TLS_PATH, VISIUM_DIR)
keep = keep_leftmost_foci(tls_xy, pitch_px, n_keep=N_KEEP_LEFT)
tls_xy, tls_bc = tls_xy[keep], tls_bc[keep]
contours = tls_smooth_contours(tls_xy, disk_r=0.70 * pitch_px, sigma=0.45 * pitch_px)
paths = sample_paths(TLS_SAMPLE)
g = geometry(TLS_SAMPLE)
pad = PAD_PITCH * pitch_px
x0 = max(0.0, float(tls_xy[:, 0].min() - pad))
y0 = max(0.0, float(tls_xy[:, 1].min() - pad))
thumb, full_w, full_h = _cohort._he_thumbnail(paths["he"], max_side=HE_MAX_SIDE)
x1 = min(float(full_w), float(tls_xy[:, 0].max() + pad))
y1 = min(float(full_h), float(tls_xy[:, 1].max() + pad))
w_um = (x1 - x0) * g["mpp_um"]
h_um = (y1 - y0) * g["mpp_um"]
print(
    f"TLS {TLS_SAMPLE}: {len(tls_bc)} spots, {len(contours)} smooth foci, "
    f"pitch {pitch_px:.1f} px, zoom [{x0:.0f}:{x1:.0f}]×[{y0:.0f}:{y1:.0f}]  "
    f"{w_um:.0f}×{h_um:.0f} µm"
)

fig, axes = plt.subplots(1, 2, figsize=(12.4, 5.8), dpi=DPI)
axes[0].imshow(thumb, extent=(0, full_w, full_h, 0), interpolation="bilinear")
draw_tls(axes[0], contours, fill=False, lw=1.6)
axes[0].add_patch(
    Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=ROI_BOX, lw=1.7, zorder=7)
)
axes[0].set_title("patient1  HE + expert TLS")
axes[0].set_xlim(0, full_w)
axes[0].set_ylim(full_h, 0)
he_roi = crop_he_rectangle(paths["he"], x0, y0, x1, y1)
axes[1].imshow(he_roi, extent=(0, he_roi.shape[1], he_roi.shape[0], 0), interpolation="bilinear")
draw_tls(
    axes[1], contours, spots=tls_xy, spot_r=spot_r_px,
    local=(x0, y0), fill=True, lw=2.2,
)
axes[1].set_title(f"TLS zoom  {w_um:.0f}×{h_um:.0f} µm")
axes[1].set_xlim(0, he_roi.shape[1])
axes[1].set_ylim(he_roi.shape[0], 0)
for ax in axes:
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
if SAVE_FIGURES:
    save_figure(fig, fig_dir / f"{TLS_SAMPLE}_TLS_HE_overview_zoom.pdf")
display(fig)
plt.close(fig)

adata = ad.read_h5ad(paths["pred"])
palettes = _cohort._palette()
xy_he = np.asarray(adata.obsm["spatial_HE"], dtype=np.float64)
in_roi = nuclei_in_rectangle(xy_he, x0, y0, x1, y1)
in_outline = nuclei_in_contours(xy_he, contours) & in_roi
print(
    f"nuclei inside black TLS outline: {int(in_outline.sum()):,} "
    f"/ zoom {int(in_roi.sum()):,}"
)
fig_z, in_roi = plot_roi_pred_spatial(
    adata,
    paths["he"],
    x0=x0,
    y0=y0,
    x1=x1,
    y1=y1,
    palettes=palettes,
    in_roi=in_roi,
    legend_mask=in_outline,
    marker_px=3.2,
    dpi=DPI,
    figsize=(16.8, 11.2),
    title=f"{TLS_SAMPLE}  Hist2Pheno in expert TLS zoom  n={int(in_roi.sum()):,}",
    save_path=None,
    save_dpi=SAVE_DPI,
)
for ax in fig_z.axes[:6]:
    draw_tls(ax, contours, local=(x0, y0), fill=False, lw=1.6)
if SAVE_FIGURES:
    save_figure(fig_z, fig_dir / f"{TLS_SAMPLE}_TLS_ROI_spatial_L1_L12_L2.pdf")
display(fig_z)
plt.close(fig_z)

fig_bar, comp = plot_pred_composition_stacked(
    adata.obs.loc[in_outline],
    palettes,
    dpi=DPI,
    figsize=(12.6, 4.6),
    title=(
        f"{TLS_SAMPLE}  Hist2Pheno inside expert TLS outline  "
        f"n={int(in_outline.sum()):,}"
    ),
)
if SAVE_FIGURES:
    save_figure(fig_bar, fig_dir / f"{TLS_SAMPLE}_TLS_outline_stacked.pdf")
    comp_path = fig_dir / f"{TLS_SAMPLE}_TLS_outline_stacked.csv"
    comp.to_csv(comp_path, index=False)
    print(comp_path)
display(fig_bar)
display(comp)
plt.close(fig_bar)


## Histology-derived niche indices (TLS / SRI / TNI)

Same implementation as `code/Xenium_lung/histology_derived_niche_index.ipynb` §3, via `Hist2Pheno_pkg/histology_niche_index.py`. Lung ARI / FRI are not used: those members (AT1/AT2, activated fibrotic FBs, SPP1+ macrophages) are not pan-cancer L2 classes.

Scores use **L2 softmax**, not the argmax label. `ensure_l2_probs` reads the five-fold `{sample}_l2_probs.npz` already next to this prediction. It does not reload a checkpoint.

| Index | Abundance | Spatial (radius R) |
|--|--|--|
| **TLS** | sum(B_cell + Plasma + CD4_T + CD8_T + Tregs + DC) | `(B_local × T_local × DC_local)^(1/3)` |
| **SRI** | (Fibroblast + Myofibroblast + Stromal) / epithelium | Fib_local / Epi_local |
| **TNI** | (fibroblast + Macrophage + Endothelial) / epithelium | `(Fib × Mac × Endo)^(1/3) / Epi_local` |

Epithelium = Epithelial, Tumor, Alveolar, Injury_epithelial, DCIS, Myoepithelial. NK is left out of TLS, as in the lung T-cell set. `spatial_HE` is native HE pixels, so R = **55 µm** is converted with the OME pixel size **0.465 µm/px** (`55 / 0.465 ≈ 118 px`), not 0.5. Hotspots are the within-slide 95th percentile. TLS, SRI, and TNI each use the same 1×3 row: abundance, spatial score without markers, then the same spatial map with cyan candidate nuclei. The two spatial panels share one color scale.


In [ ]:
import importlib

for _p in (REPO / "code" / "Hist2Pheno_pkg", REPO / "code" / "Xenium_lung"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import histology_niche_index as hni
importlib.reload(hni)
import histology_derived_niche_index as hdni
importlib.reload(hdni)
from histology_niche_index import OrganNicheConfig

import anndata as ad

TLS_RADIUS_UM = 55.0  # lung notebook cross-dataset TLS radius
RECOMPUTE_NICHE = True  # 2026.10.28, legacy niche cache lacks provenance; recompute from verified current probabilities

NPC_NICHE = OrganNicheConfig(
    pan_organ="pancancer_npc",
    tls_b=("B_cell", "Plasma"),
    tls_t=("CD4_T", "CD8_T", "Tregs"),
    tls_dc=("DC",),
    epithelium=(
        "Epithelial", "Tumor", "Alveolar",
        "Injury_epithelial", "DCIS", "Myoepithelial",
    ),
    fibroblast=("Fibroblast", "Myofibroblast", "Stromal"),
    endothelial=("Endothelial",),
    macrophage=("Macrophage",),
    um_per_he_pixel=0.4650793650793653,
    spatial_radius_um=TLS_RADIUS_UM,
    cohort="NPC",
)
display(hni.index_formula_table(NPC_NICHE))


In [ ]:
import numpy as np
niche_frames = {}
for sample in samples:
    paths = sample_paths(sample)
    niche_path = paths["pred"].with_name(f"{sample}_niche_indices.h5ad")
    mpp = float(geometry(sample)["mpp_um"])
    if niche_path.is_file() and not RECOMPUTE_NICHE:
        cached = ad.read_h5ad(niche_path)
        df = cached.obs.copy()
        xy = np.asarray(cached.obsm["spatial_HE"], dtype=np.float64)
        df["coord_x"] = xy[:, 0]
        df["coord_y"] = xy[:, 1]
        print(f"[niche] cache {niche_path.name}  n={len(df):,}  µm/px={mpp:.4f}")
    else:
        prob_path = paths["pred"].with_name(f"{sample}_l2_probs.npz")
        with np.load(prob_path, allow_pickle=True) as z:
            probs = np.asarray(z["probs"], dtype=np.float32)
            class_names = [str(x) for x in z["class_names"]]
        missing = hni.missing_index_cell_types(class_names, NPC_NICHE)
        if missing:
            raise RuntimeError(f"{sample} missing niche compartments: {missing}")
        pred = ad.read_h5ad(paths["pred"])
        coords = np.asarray(pred.obsm["spatial_HE"], dtype=np.float64)
        df = pred.obs.copy()
        df["coord_x"] = coords[:, 0]
        df["coord_y"] = coords[:, 1]
        df = hni.add_abundance_indices(df, probs, class_names, NPC_NICHE)
        df = hni.add_spatial_indices(
            df, probs, class_names, coords, NPC_NICHE,
            radius_um=TLS_RADIUS_UM,
            um_per_pixel=mpp,
        )
        out = ad.AnnData(obs=df.drop(columns=["coord_x", "coord_y"]))
        out.obsm["spatial_HE"] = coords
        out.uns["tls_radius_um"] = TLS_RADIUS_UM
        out.uns["um_per_px"] = mpp
        # Ensemble predictions have no single-checkpoint pipeline stamp, so the verified publisher cannot attach this file.
        out.write_h5ad(niche_path)
        print(f"[niche] wrote {niche_path}  n={len(df):,}")
    niche_frames[sample] = df
    print(f"  TLS hotspot: {hni.summarize_tls_hotspots(df)}")
    print(
        f"  mean TLS={df['idx_TLS'].mean():.3f}  TLS_spatial={df['idx_TLS_spatial'].mean():.3f}  "
        f"SRI={df['idx_SRI_ratio'].mean():.3f}  TNI={df['idx_TNI_ratio'].mean():.3f}"
    )


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

fig_dir = PRED_ROOT / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

quant_rows = [tls_score_row(df, sample) for sample, df in niche_frames.items()]
tls_quant = pd.DataFrame(quant_rows)
tls_quant_path = fig_dir / "tls_score_summary.csv"
tls_quant.to_csv(tls_quant_path, index=False)
print(tls_quant_path)
display(tls_quant.round(3))

for sample, df in niche_frames.items():
    fig, axes = hdni.plot_tls_abundance_vs_spatial(
        df,
        sample,
        radius_um=TLS_RADIUS_UM,
        use_entropy=False,
        split_spatial_candidates=True,
        dpi=DPI,
        save_dpi=SAVE_DPI,
        point_size=0.15,
        candidate_size=1.2,
        save_path=None,
    )
    if SAVE_FIGURES:
        save_figure(fig, fig_dir / f"{sample}_TLS_abundance_vs_spatial.png")
    display(fig)
    plt.close(fig)

    for abundance_col, spatial_col, candidate_col, tag in (
        ("idx_SRI_ratio", "idx_SRI_spatial", "sri_candidate", "SRI"),
        ("idx_TNI_ratio", "idx_TNI_spatial", "tni_candidate", "TNI"),
    ):
        fig, axes = hdni.plot_tls_abundance_vs_spatial(
            df,
            sample,
            radius_um=TLS_RADIUS_UM,
            abundance_col=abundance_col,
            spatial_col=spatial_col,
            candidate_col=candidate_col,
            use_entropy=False,
            split_spatial_candidates=True,
            score_name=tag,
            dpi=DPI,
        save_dpi=SAVE_DPI,
        point_size=0.15,
            candidate_size=1.2,
            save_path=None,
        )
        if SAVE_FIGURES:
            save_figure(fig, fig_dir / f"{sample}_{tag}_abundance_vs_spatial.png")
        display(fig)
        plt.close(fig)
